In [ ]:
import os

def count_jpgs(root_dir):
    count = 0
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(".jpg"):
                count += 1
    return count

if __name__ == "__main__":
    folder = "train"  # change if needed
    total = count_jpgs(folder)
    print(f"Total JPG images: {total}")
print("Looking at random sample of this many images")

In [ ]:
# Look at some paths

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ExifTags
from tqdm import tqdm
import matplotlib.pyplot as plt

ROOT = Path("train")

image_paths = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_paths.extend(ROOT.rglob(ext))

image_paths = sorted(image_paths)
print(f"Found {len(image_paths)} images")
print("First few:")
for p in image_paths[:5]:
    print(p)

In [ ]:
# Get familiar with "scope" and size of of images

In [ ]:
rows = []
for p in tqdm(image_paths):
    rows.append({
        "path": str(p),
        "pid": p.parent.name,
        "filename": p.name,
        "suffix": p.suffix.lower(),
        "file_size_bytes": p.stat().st_size,
    })

df_files = pd.DataFrame(rows)
df_files.head()

In [ ]:
df_files["file_size_bytes"].mean()

In [ ]:
# Image resolution, aspect ratio, mode, format

In [ ]:
rows = []

for p in tqdm(image_paths):
    try:
        with Image.open(p) as img:
            w, h = img.size
            rows.append({
                "path": str(p),
                "pid": p.parent.name,
                "width": w,
                "height": h,
                "resolution": f"{w}x{h}",
                "aspect_ratio": w / h if h else np.nan,
                "mode": img.mode,
                "format": img.format,
            })
    except Exception as e:
        rows.append({
            "path": str(p),
            "pid": p.parent.name,
            "width": np.nan,
            "height": np.nan,
            "resolution": None,
            "aspect_ratio": np.nan,
            "mode": None,
            "format": None,
            "error": str(e),
        })

df_img = pd.DataFrame(rows)
df_img.head()

In [ ]:
print("Unique resolutions:", df_img["resolution"].nunique())
print("\nMost common resolutions:")
print(df_img["resolution"].value_counts().head(20))

print("\nUnique image modes:")
print(df_img["mode"].value_counts(dropna=False))

print("\nUnique formats:")
print(df_img["format"].value_counts(dropna=False))

In [ ]:
# Common Metrics

In [ ]:
import cv2

metric_rows = []

for p in tqdm(image_paths):
    try:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        # brightness
        brightness = gray.mean()

        # contrast
        contrast = gray.std()

        # blur / sharpness proxy
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()

        # colorfulness metric
        (B, G, R) = cv2.split(img_bgr)
        rg = np.abs(R.astype(float) - G.astype(float))
        yb = np.abs(0.5 * (R.astype(float) + G.astype(float)) - B.astype(float))
        colorfulness = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2)

        metric_rows.append({
            "path": str(p),
            "pid": p.parent.name,
            "brightness": brightness,
            "contrast": contrast,
            "laplacian_var": lap_var,
            "colorfulness": colorfulness,
            "mean_r": img_rgb[:, :, 0].mean(),
            "mean_g": img_rgb[:, :, 1].mean(),
            "mean_b": img_rgb[:, :, 2].mean(),
        })
    except Exception:
        pass

df_metrics = pd.DataFrame(metric_rows)
df_metrics.head()

In [ ]:
df_metrics.describe()

In [ ]:
"""
Dataset is grayscale + highly standardized (brightness [0-255], contrast [0-127.5], rgb values [0-255] all in pretty tight ranges)
Laplacian variance [sharpness range] range is generally 0 to a few thousand; we have high variation from min-max and quarties, so range of sharpness
"""

In [ ]:
# Plot example high/low sharpness

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# sort by sharpness
df_sorted = df_metrics.sort_values("laplacian_var")

N = 12  # number of images to show

low_sharp = df_sorted.head(N)
high_sharp = df_sorted.tail(N)

print("Low sharpness range:", low_sharp["laplacian_var"].min(), "→", low_sharp["laplacian_var"].max())
print("High sharpness range:", high_sharp["laplacian_var"].min(), "→", high_sharp["laplacian_var"].max())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5))
axes = axes.flatten()
low_sharp = low_sharp[:4]
for ax, (_, row) in zip(axes, low_sharp.iterrows()):
    img = Image.open(row["path"]).convert("L")  # ensure grayscale
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{row['laplacian_var']:.1f}", fontsize=9)
    ax.axis("off")

for ax in axes[len(low_sharp):]:
    ax.axis("off")

plt.suptitle("LOW Sharpness (Blurry Images)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5))
axes = axes.flatten()
high_sharp = high_sharp[:4]
for ax, (_, row) in zip(axes, high_sharp.iterrows()):
    img = Image.open(row["path"]).convert("L")
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{row['laplacian_var']:.1f}", fontsize=9)
    ax.axis("off")

for ax in axes[len(high_sharp):]:
    ax.axis("off")

plt.suptitle("HIGH Sharpness (Detailed / Noisy Images)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mid = df_sorted.iloc[len(df_sorted)//2 - N//2 : len(df_sorted)//2 + N//2]
mid = mid[:4]
fig, axes = plt.subplots(2, 2, figsize=(8, 5))
axes = axes.flatten()

for ax, (_, row) in zip(axes, mid.iterrows()):
    img = Image.open(row["path"]).convert("L")
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{row['laplacian_var']:.1f}", fontsize=9)
    ax.axis("off")

plt.suptitle("MID Sharpness (Typical Images)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
## PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

MAX_IMAGES = min(3000, len(image_paths))   # increase if you want

sample_paths = image_paths[:MAX_IMAGES]

X = []
meta = []

for p in tqdm(sample_paths):
    try:
        img = Image.open(p).convert("L")#.#resize(RESIZE_TO)
        arr = np.asarray(img, dtype=np.float32).flatten() / 255.0
        X.append(arr)
        meta.append({"path": str(p), "pid": p.parent.name})
    except Exception:
        pass

X = np.array(X)
df_embed = pd.DataFrame(meta)

print("Embedding matrix shape:", X.shape)

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_embed["pca1"] = X_pca[:, 0]
df_embed["pca2"] = X_pca[:, 1]

print("Explained variance ratio:", pca.explained_variance_ratio_)
df_embed.head()

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(df_embed["pca1"], df_embed["pca2"], s=6, alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of images")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

MAX_IMAGES = min(3000, len(image_paths))   # adjust if needed
RESIZE_TO = (32, 32)

sample_paths = image_paths[:MAX_IMAGES]

X = []
meta = []

for p in tqdm(sample_paths):
    try:
        img = Image.open(p).convert("L").resize(RESIZE_TO)
        arr = np.asarray(img, dtype=np.float32).flatten() / 255.0
        X.append(arr)
        meta.append({
            "path": str(p),
            "pid": p.parent.name,
        })
    except Exception:
        pass

X = np.array(X)
df_cluster = pd.DataFrame(meta)

print("X shape:", X.shape)
df_cluster.head()

In [ ]:
from sklearn.cluster import KMeans

K = 5  # try 4, 5, 6, 8, etc.

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

df_cluster["cluster"] = labels
df_cluster["cluster"].value_counts().sort_index()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

df_cluster["pca1"] = X_pca[:, 0]
df_cluster["pca2"] = X_pca[:, 1]

plt.figure(figsize=(7, 6))
for c in sorted(df_cluster["cluster"].unique()):
    subset = df_cluster[df_cluster["cluster"] == c]
    plt.scatter(subset["pca1"], subset["pca2"], s=8, alpha=0.6, label=f"Cluster {c}")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of images colored by KMeans cluster")
plt.legend()
plt.show()

In [ ]:
centers = kmeans.cluster_centers_

fig, axes = plt.subplots(1, K, figsize=(3*K, 3))

if K == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    center_img = centers[i].reshape(RESIZE_TO[1], RESIZE_TO[0])
    ax.imshow(center_img, cmap="gray")
    ax.set_title(f"Cluster {i}")
    ax.axis("off")

plt.suptitle("KMeans cluster centers", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

IMAGES_PER_CLUSTER = 3

clusters = sorted(df_cluster["cluster"].unique())

fig, axes = plt.subplots(len(clusters), IMAGES_PER_CLUSTER, 
                         figsize=(4 * IMAGES_PER_CLUSTER, 3 * len(clusters)))

# handle case when only 1 cluster
if len(clusters) == 1:
    axes = [axes]

for i, c in enumerate(clusters):
    subset = df_cluster[df_cluster["cluster"] == c].head(IMAGES_PER_CLUSTER)

    for j in range(IMAGES_PER_CLUSTER):
        ax = axes[i][j] if len(clusters) > 1 else axes[j]

        if j < len(subset):
            row = subset.iloc[j]
            img = Image.open(row["path"]).convert("L")
            ax.imshow(img, cmap="gray")
            ax.set_title(f"C{c}", fontsize=10)
        else:
            ax.axis("off")

        ax.axis("off")

plt.suptitle("3 Images per KMeans Cluster", fontsize=16)
plt.tight_layout()
plt.show()